In [ ]:
!pip install transformers
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, TrainingArguments, Trainer, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv')

In [ ]:
train_data.head(5)

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop = True)
val_data = train_data.sample(n=500, random_state=42).reset_index(drop = True)

In [ ]:
train_data.shape

In [ ]:
val_data.shape

# Data pre_processing

In [ ]:
import re 

def clean_data(text):
    text = str(text)
    text = re.sub(r"\r\n", " ", text) #Lines
    text = re.sub(r"\s+", " ", text) #Spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower()

    return text

In [ ]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [ ]:
train_data['dialogue'][0]

# Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation=True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation=True)

    inputs['labels'] = targets['input_ids']     # token_ids => add to input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [ ]:
train_dataset[0]

In [ ]:
# input_ids - dialogue => token ids
# attention mask
# labels - target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

In [ ]:
len(train_dataset[0]["labels"])

In [ ]:
type(train_dataset)
type(val_dataset)

# Working with our model

In [ ]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif  torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/Datasets/Text_Summarizer",
    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps=500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
trainer.train()